# Volatility-Managed Portfolios — narrative walkthrough

Replication & extension of **Moreira & Muir (2017)**.

> *You can't predict the **price**. But you can predict the **risk** — and that's enough.*

This notebook runs the whole pipeline phase by phase. Run it from the repo root
(so that `import src...` resolves), with the project virtual-env kernel.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))

import pandas as pd
from src.data_loader import load_dataset, describe
from src import config
config.set_plot_style()

daily, monthly = load_dataset(rebuild=True)
print(daily.shape, monthly.shape, daily.index.min().date(), '->', daily.index.max().date())
describe(daily, monthly).round(4)

## Phase 1 — The null result

Predicting next month's **return** out-of-sample fails ($R^2 \le 0$). Predicting
next month's **volatility** works ($R^2 \approx 0.3$). That asymmetry is the thesis.

In [ ]:
from src.predictability import run_predictability
from src import plots

pred = run_predictability(monthly)
display(pred['table'].round(4))
plots.fig_predictability_r2(pred['table'], config.FIGURES / 'fig01_predictability_r2.png')
from IPython.display import Image; Image(str(config.FIGURES / 'fig01_predictability_r2.png'))

## Phase 2 — Forecasting volatility

Naive / GARCH(1,1) / HAR / ensemble, walk-forward. Judged by RMSE **and** QLIKE,
with Diebold-Mariano significance tests.

In [ ]:
from src.vol_forecast import run_vol_forecasts
vf = run_vol_forecasts(daily, monthly, verbose=False)
print('Eval:', vf['eval_start'].date(), '->', vf['eval_end'].date(), f"({vf['n_eval']} months)")
display(vf['metrics'].round(5))
display(vf['dm_table'].round(4))

## Phases 3 & 4 — Strategy and honest evaluation

$f_{managed}[t] = (c/\hat\sigma^2[t])\, f[t]$, vol-matched to buy-and-hold, with a
2× leverage cap. The headline test is $\alpha$ in `managed = α + β·market + ε`.

In [ ]:
from src.strategy import build_managed_strategy
from src import evaluation as ev

strat, info = build_managed_strategy(monthly, vf['forecasts']['naive'], leverage_cap=2.0)
mm = ev.performance_metrics(strat['managed'], 'managed')
bh = ev.performance_metrics(strat['f'], 'buy_and_hold')
alpha = ev.alpha_regression(strat['managed'], strat['f'])
display(pd.DataFrame([bh, mm]).set_index('name').T.round(3))
print('alpha (ann): {:.2%}  t={:.2f}  appraisal={:.2f}'.format(alpha['alpha_annual'], alpha['alpha_tstat'], alpha['appraisal_ratio']))
print('Deflated Sharpe:', round(ev.deflated_sharpe_ratio(strat['managed'], 3)['deflated_sharpe'], 3))
plots.fig_equity_curve(strat, config.FIGURES / 'fig05_equity_curve.png')
from IPython.display import Image; Image(str(config.FIGURES / 'fig05_equity_curve.png'))

In [ ]:
# Forecaster comparison, cost & leverage robustness
display(ev.compare_forecasters(monthly, vf['forecasts'], ['naive','har','garch','ensemble']).round(3))
display(ev.cost_sensitivity(strat).round(3))
print('Break-even cost:', round(ev.breakeven_cost(strat), 1), 'bps')
display(ev.subperiod_analysis(strat).round(3))

## Phase 5 — Regimes (HMM)

A Gaussian HMM, given only returns and **no crisis dates**, rediscovers the
turbulent regimes. Smoothed states are for the figure; the *filtered* signal is
the only thing admissible inside a backtest.

In [ ]:
from src import regimes as rg
hmm = rg.fit_hmm_smoothed(monthly, n_states=2)
display(hmm['state_stats'].round(3))
display(rg.regime_decomposition(strat, hmm['states'], hmm['labels']).round(3))
plots.fig_regimes(monthly, hmm, config.FIGURES / 'fig07_regimes.png')
from IPython.display import Image; Image(str(config.FIGURES / 'fig07_regimes.png'))